<a href="https://colab.research.google.com/github/gabrielcorlett095-beep/air-monitor-project/blob/main/monitor_de_qualidade_do_ar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit pyngrok pandas requests psycopg2-binary

In [ ]:
!mkdir -p .streamlit

%%writefile .streamlit/secrets.toml
[postgres]
host = "dpg-d8vgb19o3t8c73b7tla0-a.oregon-postgres.render.com"
port = "5432"
database = "qualidade_ar_yks1"
user = "qualidade_ar_yks1_user"
password = "73xewecHyB8c8RduY2NhBSuRG8QPKhP1"

In [ ]:
%%writefile app.py
import streamlit as st
import requests
import pandas as pd
import psycopg2 # Importação do conector do banco

# Configuração da página
st.set_page_config(page_title="Monitor de Ar Pro", page_icon="🌍", layout="centered")

def get_color_style(param, value):
    if param == 'pm2_5':
        if value <= 12: return 'mediumseagreen', 'white'
        elif value <= 35: return 'gold', 'black'
        else: return 'tomato', 'white'
    elif param == 'nitrogen_dioxide':
        if value <= 40: return 'mediumseagreen', 'white'
        elif value <= 120: return 'gold', 'black'
        else: return 'tomato', 'white'
    elif param == 'ozone':
        if value <= 100: return 'mediumseagreen', 'white'
        elif value <= 160: return 'gold', 'black'
        else: return 'tomato', 'white'
    return 'gray', 'white'

# --- FUNÇÃO DO CRUD (CREATE) ---
def salvar_no_banco(cidade, pm25, no2, o3):
    try:
        # Conecta ao banco lendo os dados do secrets.toml
        conn = psycopg2.connect(
            host=st.secrets["postgres"]["host"],
            database=st.secrets["postgres"]["database"],
            user=st.secrets["postgres"]["user"],
            password=st.secrets["postgres"]["password"],
            port=st.secrets["postgres"]["port"]
        )
        cursor = conn.cursor()

        # 1. Cria a tabela no Render automaticamente caso ela não exista
        tabela_sql = """
        CREATE TABLE IF NOT EXISTS historico_pesquisas (
            id SERIAL PRIMARY KEY,
            cidade VARCHAR(100),
            pm2_5 NUMERIC,
            nitrogen_dioxide NUMERIC,
            ozone NUMERIC,
            data_hora TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );
        """
        cursor.execute(tabela_sql)

        # 2. Comando SQL de inserção
        sql = "INSERT INTO historico_pesquisas (cidade, pm2_5, nitrogen_dioxide, ozone) VALUES (%s, %s, %s, %s)"
        val = (cidade, pm25, no2, o3)

        cursor.execute(sql, val)
        conn.commit()

        cursor.close()
        conn.close()
        return True
    except Exception as e:
        st.error(f"Erro na conexão com o banco de dados: {e}")
        return False
# -------------------------------

st.title("🌍 Monitor de Qualidade do Ar Pro")
st.write("Verifique o nível de poluição atual, localização no mapa e salve no banco de dados.")

city = st.text_input("📍 Digite o nome da cidade:", placeholder="Ex: Natal, Nova York, Tokyo")

if st.button("Verificar Qualidade", type="primary"):
    if not city:
        st.warning("Por favor, digite o nome de uma cidade.")
    else:
        with st.spinner(f"Buscando dados para '{city}'..."):
            geo_url = "https://geocoding-api.open-meteo.com/v1/search"
            params_geo = {"name": city, "count": 1, "language": "pt"}

            try:
                geo_res = requests.get(geo_url, params=params_geo)

                if geo_res.status_code != 200:
                    st.error(f"Erro ao buscar cidade (Status {geo_res.status_code}).")
                else:
                    geo_response = geo_res.json()

                    if 'results' not in geo_response or not geo_response['results']:
                        st.error(f"Não consegui encontrar as coordenadas para '{city}'.")
                    else:
                        primeiro_resultado = geo_response['results'][0]
                        lat = primeiro_resultado['latitude']
                        lon = primeiro_resultado['longitude']
                        city_name = primeiro_resultado['name']
                        country = primeiro_resultado.get('country', '')

                        st.success(f"Dados carregados para **{city_name} ({country})**")

                        aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
                        params_aq = {
                            "latitude": lat,
                            "longitude": lon,
                            "current": "pm2_5,nitrogen_dioxide,ozone",
                            "hourly": "pm2_5,nitrogen_dioxide,ozone",
                            "past_days": 1
                        }

                        aq_res = requests.get(aq_url, params=params_aq)

                        if aq_res.status_code != 200:
                            st.error(f"A API de Qualidade do Ar recusou a conexão. Resposta: {aq_res.text}")
                        else:
                            aq_response = aq_res.json()

                            st.subheader("📊 Condições Atuais")
                            current_data = aq_response.get('current', {})
                            valores = {
                                'pm2_5': current_data.get('pm2_5'),
                                'nitrogen_dioxide': current_data.get('nitrogen_dioxide'),
                                'ozone': current_data.get('ozone')
                            }
                            nomes_exibicao = {'pm2_5': 'PM2.5', 'nitrogen_dioxide': 'NO₂', 'ozone': 'O₃'}

                            cols = st.columns(3)
                            for (param, value), col in zip(valores.items(), cols):
                                if value is not None:
                                    bg_color, font_color = get_color_style(param, value)
                                    sigla = nomes_exibicao[param]

                                    html_card = f"""
                                    <div style='padding: 20px; border-radius: 8px; background-color: {bg_color}; color: {font_color}; text-align: center; box-shadow: 2px 2px 8px rgba(0,0,0,0.1); margin-bottom: 10px;'>
                                        <div style='font-size: 24px; font-weight: bold; margin-bottom: 5px;'>{sigla}</div>
                                        <div style='font-size: 18px;'>{value} <span style='font-size: 14px;'>µg/m³</span></div>
                                    </div>
                                    """
                                    col.markdown(html_card, unsafe_allow_html=True)

                            # --- CHAMADA DO CRUD AQUI ---
                            if salvar_no_banco(city_name, valores['pm2_5'], valores['nitrogen_dioxide'], valores['ozone']):
                                st.success("💾 Histórico salvo no banco de dados do Render com sucesso!")
                            # ----------------------------

                            st.subheader("📈 Histórico das Últimas 24 Horas")
                            hourly_data = aq_response.get('hourly', {})
                            if hourly_data:
                                df_hourly = pd.DataFrame({
                                    'Horário': pd.to_datetime(hourly_data.get('time')),
                                    'PM2.5': hourly_data.get('pm2_5'),
                                    'NO₂': hourly_data.get('nitrogen_dioxide'),
                                    'O₃': hourly_data.get('ozone')
                                })
                                df_last_24h = df_hourly.tail(24).set_index('Horário')
                                st.line_chart(df_last_24h)

                            st.subheader("🗺️ Localização")
                            map_data = pd.DataFrame({'lat': [lat], 'lon': [lon]})
                            st.map(map_data, zoom=10)

            except Exception as e:
                st.error(f"Erro inesperado no processamento: {e}")

In [ ]:
from pyngrok import ngrok

# Configure seu NOVO Token de Autenticação do Ngrok (Gere um novo no site!)
ngrok.set_auth_token("COLOQUE_SEU_NOVO_TOKEN_AQUI")

# Limpa conexões travadas antigas
ngrok.kill()

# Cria o link de acesso seguro e estável
public_url = ngrok.connect(8501)
print("👉 LINK DO SEU MONITOR:", public_url.public_url)

# Executa o Streamlit em segundo plano
!streamlit run app.py --server.port 8501 &>/dev/null &